# **BiLSTM Model**
 For Named Entity Recognition (NER) using the 30 manually created outlines dataset

In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/Colab Notebooks/FYP /OOP Annotated.zip'
extract_path = '/content/extracted_data'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Dataset extracted to: {extract_path}")

Dataset extracted to: /content/extracted_data


In [ ]:
# ========================
# STEP 1: Imports
# ========================
import os
import zipfile
import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, TimeDistributed, Dense, Masking

# ========================
# STEP 2: Extract Dataset
# ========================
zip_path = '/content/drive/MyDrive/Colab Notebooks/FYP /OOP Annotated.zip'
extract_path = '/content/extracted_data'
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print(f"Dataset extracted to: {extract_path}")

# ========================
# STEP 3: Load Annotations
# ========================
def load_spacy_annotations(base_path):
    sentences, labels, tag_set = [], [], set()
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if "annotations (1)" in file and file.endswith(".json"):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        data = json.load(f)
                    if "annotations" not in data:
                        continue
                    for text, ann in data["annotations"]:
                        if not text.strip():
                            continue
                        tokens = text.strip().split()
                        token_labels = ["O"] * len(tokens)
                        for start, end, label in ann.get("entities", []):
                            span_text = text[start:end]
                            for i, token in enumerate(tokens):
                                if span_text in token:
                                    token_labels[i] = label
                                    tag_set.add(label)
                        sentences.append(tokens)
                        labels.append(token_labels)
                except Exception as e:
                    print(f"[WARN] Skipping {file_path}: {e}")
    return sentences, labels, tag_set

base_path = "/content/extracted_data/OOP Annotated"
sentences, labels, tag_set = load_spacy_annotations(base_path)

print("Total sentences:", len(sentences))
print("Sample sentence:", sentences[0])
print("Sample labels:", labels[0])
print("Tags found:", tag_set)

# ========================
# STEP 4: Preprocessing
# ========================
all_words = set([w for s in sentences for w in s])
all_tags = sorted(list(set([t for l in labels for t in l])))

word2idx = {w: i+2 for i, w in enumerate(sorted(all_words))}
word2idx["PAD"] = 0
word2idx["UNK"] = 1

tag2idx = {t: i+1 for i, t in enumerate(all_tags)}  # start from 1
tag2idx["PAD"] = 0  # mask index must be 0

idx2tag = {i: t for t, i in tag2idx.items()}

X = [[word2idx.get(w, word2idx["UNK"]) for w in s] for s in sentences]
y = [[tag2idx[t] for t in l] for l in labels]

MAXLEN = max(len(s) for s in X)
X = pad_sequences(X, maxlen=MAXLEN, padding="post", value=word2idx["PAD"])
y = pad_sequences(y, maxlen=MAXLEN, padding="post", value=tag2idx["PAD"])

print("Vocabulary size:", len(word2idx))
print("Number of tags:", len(tag2idx))
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

# ========================
# STEP 5: Train/Test Split
# ========================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ========================
# STEP 6: Build BiLSTM Model
# ========================
EMBEDDING_DIM = 128
LSTM_UNITS = 64
NUM_CLASSES = len(tag2idx)  # include PAD

model = Sequential([
    Embedding(input_dim=len(word2idx), output_dim=EMBEDDING_DIM, mask_zero=True),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)),
    TimeDistributed(Dense(NUM_CLASSES, activation="softmax"))
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.build(input_shape=(None, MAXLEN))
model.summary()

# ========================
# STEP 7: Train Model
# ========================
history = model.fit(
    X_train, np.expand_dims(y_train, -1),
    validation_data=(X_test, np.expand_dims(y_test, -1)),
    batch_size=32,
    epochs=40,
    verbose=1
)

# ========================
# STEP 8: Evaluation
# ========================
y_pred = model.predict(X_test)

y_true_labels, y_pred_labels = [], []

for i in range(len(y_test)):
    for j in range(len(y_test[i])):
        if y_test[i][j] != tag2idx["PAD"]:
            y_true_labels.append(idx2tag[y_test[i][j]])
            y_pred_labels.append(idx2tag[np.argmax(y_pred[i][j])])

print(classification_report(y_true_labels, y_pred_labels))


Dataset extracted to: /content/extracted_data
Total sentences: 486
Sample sentence: ['Function-Oriented', 'Programming']
Sample labels: ['O', 'O']
Tags found: {'METHOD', 'TOPIC', 'CONCEPT', 'OTHERS'}
Vocabulary size: 1188
Number of tags: 6
Shape of X: (486, 58)
Shape of y: (486, 58)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 58, 128)        │       152,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 58, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 58, 6)          │           774 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 251,654 (983.02 KB)

 Trainable params: 251,654 (983.02 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 19s 510ms/step - accuracy: 0.2903 - loss: 1.7391 - val_accuracy: 0.1302 - val_loss: 1.4144
Epoch 2/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 206ms/step - accuracy: 0.1166 - loss: 1.1694 - val_accuracy: 0.1302 - val_loss: 0.5936
Epoch 3/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 209ms/step - accuracy: 0.1129 - loss: 0.4914 - val_accuracy: 0.1302 - val_loss: 0.4827
Epoch 4/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 341ms/step - accuracy: 0.1103 - loss: 0.5411 - val_accuracy: 0.1302 - val_loss: 0.4055
Epoch 5/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 212ms/step - accuracy: 0.1104 - loss: 0.3820 - val_accuracy: 0.1302 - val_loss: 0.3580
Epoch 6/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 203ms/step - accuracy: 0.1136 - loss: 0.3099 - val_accuracy: 0.1302 - val_loss: 0.3315
Epoch 7/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 202ms/step - accuracy: 0.1184 - loss: 0.2392 - val_accuracy: 0.1302 - val_loss: 0.3147
Epoch 8/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 207ms/step - accuracy: 0.1169 - loss: 0.2103 - val_accuracy: 0